In [1]:
# ============================================================
# CELL 1: Unzip & Parse Cricsheet JSON files into DataFrame
# ============================================================

import zipfile, json, os
import pandas as pd
from pathlib import Path

# ---- Step 1: Unzip all three datasets ----
zip_files = [
    "icc_mens_t20_world_cup_json.zip",
    "psl_json.zip",
    "t20s_male_json.zip"
]

extract_dir = "all_matches"
os.makedirs(extract_dir, exist_ok=True)

for zf in zip_files:
    if os.path.exists(zf):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(extract_dir)
        print(f"Extracted: {zf}")
    else:
        print(f"WARNING: {zf} not found, skipping.")

# ---- Step 2: Collect all .json file paths ----
all_json_files = list(Path(extract_dir).rglob("*.json"))
print(f"\nTotal JSON match files found: {len(all_json_files)}")

# ---- Step 3: Parse each match file ball by ball ----
def parse_match(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)

    info = data.get('info', {})
    innings_list = data.get('innings', [])

    # We only care about T20 matches with a clear winner
    outcome = info.get('outcome', {})
    winner = outcome.get('winner', None)
    teams = info.get('teams', [])

    if not winner or len(teams) < 2:
        return []

    rows = []
    match_id = str(filepath.stem)

    for inning_idx, inning in enumerate(innings_list):
        batting_team = inning.get('team', '')
        bowling_team = [t for t in teams if t != batting_team]
        bowling_team = bowling_team[0] if bowling_team else ''

        overs = inning.get('overs', [])

        runs_so_far = 0
        wickets_so_far = 0
        ball_number = 0  # ball index within innings (0..119)

        for over_data in overs:
            over_num = over_data.get('over', 0)
            deliveries = over_data.get('deliveries', [])

            for delivery in deliveries:
                runs_obj = delivery.get('runs', {})
                batter_runs = runs_obj.get('batter', 0)
                total_runs = runs_obj.get('total', 0)
                extras = runs_obj.get('extras', 0)

                wickets = delivery.get('wickets', [])
                is_wicket = 1 if len(wickets) > 0 else 0

                # Label next-ball outcome: 0=dot/other, 1=boundary(4 or 6), 2=wicket
                if is_wicket:
                    next_ball_label = 2
                elif batter_runs >= 4:
                    next_ball_label = 1
                else:
                    next_ball_label = 0

                rows.append({
                    'match_id':       match_id,
                    'inning':         inning_idx + 1,
                    'batting_team':   batting_team,
                    'bowling_team':   bowling_team,
                    'winner':         winner,
                    'batting_won':    int(batting_team == winner),  # target label
                    'ball_number':    ball_number,
                    'over':           over_num,
                    'runs_so_far':    runs_so_far,
                    'wickets_so_far': wickets_so_far,
                    'batter_runs':    batter_runs,
                    'total_runs_ball':total_runs,
                    'is_wicket':      is_wicket,
                    'next_ball_label':next_ball_label,
                })

                runs_so_far    += total_runs
                wickets_so_far += is_wicket
                ball_number    += 1

    return rows

# ---- Step 4: Build the full DataFrame ----
all_rows = []
failed = 0
for fp in all_json_files:
    try:
        rows = parse_match(fp)
        all_rows.extend(rows)
    except Exception as e:
        failed += 1

df = pd.DataFrame(all_rows)
print(f"\nParsed {len(df)} ball-by-ball rows from {len(all_json_files) - failed} matches ({failed} failed).")
print(df.head(10))
print("\nColumns:", df.columns.tolist())

Extracted: icc_mens_t20_world_cup_json.zip
Extracted: psl_json.zip
Extracted: t20s_male_json.zip

Total JSON match files found: 3723

Parsed 825793 ball-by-ball rows from 3723 matches (0 failed).
  match_id  inning batting_team bowling_team     winner  batting_won  \
0  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
1  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
2  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
3  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
4  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
5  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
6  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
7  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
8  1001349       1    Australia    Sri Lanka  Sri Lanka            0   
9  1001349       1    Australia    Sri Lanka  Sri Lanka            0   

   ball_num

In [2]:
# ============================================================
# CELL 2: Feature Engineering & Sequence Creation
# ============================================================

import numpy as np

MAX_BALLS = 120       # max balls per innings (T20 = 20 overs x 6)
MAX_SCORE = 260.0     # normalization ceiling for scores
MAX_WICKETS = 10.0

# ---- Step 1: Add features per ball ----

# Balls remaining
df['balls_remaining'] = MAX_BALLS - df['ball_number']

# Current run rate (runs per ball)
df['run_rate'] = df['runs_so_far'] / (df['ball_number'] + 1)

# Required run rate — only meaningful in 2nd innings
# For 1st innings set to 0, for 2nd innings compute against target
# We'll compute final innings scores to derive targets
innings_final = (
    df.groupby(['match_id', 'inning'])['runs_so_far'].max()
      .reset_index()
      .rename(columns={'runs_so_far': 'final_score'})
)
df = df.merge(innings_final, on=['match_id', 'inning'], how='left')

# For 2nd innings: target = 1st innings final score + 1
first_innings_scores = (
    innings_final[innings_final['inning'] == 1]
    .rename(columns={'final_score': 'target_score', 'inning': '_drop'})
    .drop(columns=['_drop'])
)
df = df.merge(first_innings_scores, on='match_id', how='left')
df['target_score'] = df['target_score'].fillna(0.0)

df['runs_needed'] = np.where(
    df['inning'] == 2,
    df['target_score'] - df['runs_so_far'],
    0.0
)
df['req_run_rate'] = np.where(
    (df['inning'] == 2) & (df['balls_remaining'] > 0),
    df['runs_needed'] / df['balls_remaining'],
    0.0
)

# ---- Step 2: Select & Normalize features ----
FEATURE_COLS = [
    'ball_number',        # progress through innings
    'runs_so_far',        # runs scored so far
    'wickets_so_far',     # wickets lost so far
    'balls_remaining',    # balls left
    'run_rate',           # current run rate
    'target_score',       # target (2nd innings only, else 0)
    'runs_needed',        # runs to win (2nd innings only)
    'req_run_rate',       # required run rate
    'inning',             # 1 or 2
]

# Simple min-max style normalization (fixed ranges for reproducibility)
df['ball_number']     = df['ball_number']     / MAX_BALLS
df['runs_so_far']     = df['runs_so_far']     / MAX_SCORE
df['wickets_so_far']  = df['wickets_so_far']  / MAX_WICKETS
df['balls_remaining'] = df['balls_remaining'] / MAX_BALLS
df['run_rate']        = df['run_rate'].clip(0, 5) / 5.0
df['target_score']    = df['target_score']    / MAX_SCORE
df['runs_needed']     = df['runs_needed'].clip(0, MAX_SCORE) / MAX_SCORE
df['req_run_rate']    = df['req_run_rate'].clip(0, 10) / 10.0
df['inning']          = (df['inning'] - 1).astype(float)  # 0 or 1

# ---- Step 3: Build sequences per (match, inning) ----
# Each sequence: shape [MAX_BALLS, num_features]
# Targets per sequence (taken from the last valid ball):
#   - win_prob_label  : 0/1 (did batting team win?)
#   - final_score     : normalized final score of this innings
#   - next_ball_labels: per-step classification (shape [MAX_BALLS])

NUM_FEATURES = len(FEATURE_COLS)
groups = df.groupby(['match_id', 'inning'])

X_sequences    = []   # [N, 120, features]
y_win          = []   # [N] — binary
y_final_score  = []   # [N] — regression (normalized)
y_next_ball    = []   # [N, 120] — 3-class per step

for (match_id, inning), group in groups:
    group = group.sort_values('ball_number').reset_index(drop=True)

    n_balls = len(group)
    if n_balls < 6:   # skip very short innings (rain, etc.)
        continue

    # Build fixed-length sequence (pad with zeros if < MAX_BALLS)
    seq_features   = np.zeros((MAX_BALLS, NUM_FEATURES), dtype=np.float32)
    seq_next_ball  = np.zeros(MAX_BALLS, dtype=np.int64)

    actual_len = min(n_balls, MAX_BALLS)
    seq_features[:actual_len] = group[FEATURE_COLS].values[:actual_len]
    seq_next_ball[:actual_len] = group['next_ball_label'].values[:actual_len]

    # Labels
    win_label   = int(group['batting_won'].iloc[0])
    final_score = float(group['final_score'].iloc[0]) / MAX_SCORE

    X_sequences.append(seq_features)
    y_win.append(win_label)
    y_final_score.append(final_score)
    y_next_ball.append(seq_next_ball)

X = np.array(X_sequences)          # [N, 120, features]
Y_win = np.array(y_win)             # [N]
Y_score = np.array(y_final_score)   # [N]
Y_next = np.array(y_next_ball)      # [N, 120]

print(f"Sequences created: {X.shape}")
print(f"Win labels: {Y_win.shape}  |  Score labels: {Y_score.shape}  |  Next-ball labels: {Y_next.shape}")
print(f"Win rate in dataset: {Y_win.mean():.2%}")
print(f"Next ball class distribution: {np.unique(Y_next, return_counts=True)}")

Sequences created: (7205, 120, 9)
Win labels: (7205,)  |  Score labels: (7205,)  |  Next-ball labels: (7205, 120)
Win rate in dataset: 49.98%
Next ball class distribution: (array([0, 1, 2]), array([709189, 111683,  43728]))


In [3]:
# ============================================================
# CELL 3: LSTM Model with 3 Output Heads (PyTorch)
# ============================================================

import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

NUM_FEATURES = X.shape[2]   # number of input features per ball
HIDDEN_SIZE  = 128           # LSTM hidden units
NUM_LAYERS   = 2             # stacked LSTM layers
DROPOUT      = 0.3
MAX_BALLS    = 120

class CricketIQ_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()

        # Shared LSTM backbone — processes the sequence of balls
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,       # input shape: [batch, seq, features]
            dropout     = dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        # --- Head 1: Win Probability ---
        # Uses the LAST hidden state (sequence-level classification)
        # Binary output: 0 = batting team loses, 1 = wins
        self.win_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)           # raw logit, BCEWithLogitsLoss applied outside
        )

        # --- Head 2: Final Score Regression ---
        # Also uses the last hidden state
        self.score_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()               # output in [0,1], scale back by MAX_SCORE
        )

        # --- Head 3: Next Ball Outcome Classification ---
        # Applied at EVERY time step (many-to-many)
        # 3 classes: 0=dot/other, 1=boundary, 2=wicket
        self.next_ball_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 3)           # CrossEntropyLoss applied outside
        )

    def forward(self, x):
        # x: [batch, 120, features]
        lstm_out, (h_n, c_n) = self.lstm(x)
        # lstm_out: [batch, 120, hidden_size]  — every step
        # h_n:      [num_layers, batch, hidden_size] — final hidden

        # Last hidden state from top LSTM layer
        last_hidden = self.dropout(h_n[-1])  # [batch, hidden_size]

        # Head 1 — Win probability (one value per sequence)
        win_logit = self.win_head(last_hidden).squeeze(-1)    # [batch]

        # Head 2 — Final score (one value per sequence)
        score_pred = self.score_head(last_hidden).squeeze(-1) # [batch]

        # Head 3 — Next ball at each step (applied to all 120 time steps)
        all_steps = self.dropout(lstm_out)                    # [batch, 120, hidden]
        next_ball_logits = self.next_ball_head(all_steps)     # [batch, 120, 3]

        return win_logit, score_pred, next_ball_logits


# ---- Instantiate and verify ----
model = CricketIQ_LSTM(
    input_size  = NUM_FEATURES,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Quick shape test
dummy = torch.zeros(4, MAX_BALLS, NUM_FEATURES).to(device)
out_win, out_score, out_next = model(dummy)
print(f"\nShape check:")
print(f"  Win logits:        {out_win.shape}")       # [4]
print(f"  Score predictions: {out_score.shape}")     # [4]
print(f"  Next ball logits:  {out_next.shape}")      # [4, 120, 3]

Using device: cpu
CricketIQ_LSTM(
  (lstm): LSTM(9, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (win_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
  (score_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
  (next_ball_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=3, bias=True)
  )
)

Total parameters: 228,357

Shape check:
  Win logits:        torch.Size([4])
  Score predictions: torch.Size([4])
  Next ball logits:  torch.Size([4, 120, 3])


In [4]:
# ============================================================
# CELL 4: Train-Test Split & Training Loop
# ============================================================

from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split

# ---- Hyperparameters ----
BATCH_SIZE  = 64
EPOCHS      = 20
LR          = 1e-3
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.10

# Loss weights to balance the three tasks
W_WIN   = 1.0
W_SCORE = 0.5
W_NEXT  = 0.5

# ---- Dataset class ----
class CricketDataset(Dataset):
    def __init__(self, X, Y_win, Y_score, Y_next):
        self.X       = torch.tensor(X,       dtype=torch.float32)
        self.Y_win   = torch.tensor(Y_win,   dtype=torch.float32)
        self.Y_score = torch.tensor(Y_score, dtype=torch.float32)
        self.Y_next  = torch.tensor(Y_next,  dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y_win[idx], self.Y_score[idx], self.Y_next[idx]

# ---- Split ----
N = len(X)
indices = list(range(N))
train_idx, temp_idx = train_test_split(indices, test_size=VAL_SPLIT + TEST_SPLIT, random_state=42)
val_idx,   test_idx = train_test_split(temp_idx, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT), random_state=42)

def make_subset(idx):
    return CricketDataset(X[idx], Y_win[idx], Y_score[idx], Y_next[idx])

train_ds = make_subset(train_idx)
val_ds   = make_subset(val_idx)
test_ds  = make_subset(test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ---- Loss functions & optimizer ----
loss_win_fn   = nn.BCEWithLogitsLoss()
loss_score_fn = nn.MSELoss()
loss_next_fn  = nn.CrossEntropyLoss(ignore_index=-1)  # ignore padded positions

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

# ---- Helper: run one epoch ----
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct_win = 0
    total_samples = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_b, Y_win_b, Y_score_b, Y_next_b in loader:
            X_b      = X_b.to(device)
            Y_win_b  = Y_win_b.to(device)
            Y_score_b= Y_score_b.to(device)
            Y_next_b = Y_next_b.to(device)

            win_logit, score_pred, next_logits = model(X_b)

            # Next-ball loss: reshape [batch*120, 3] vs [batch*120]
            next_logits_flat = next_logits.view(-1, 3)
            next_labels_flat = Y_next_b.view(-1)

            loss_w = loss_win_fn(win_logit, Y_win_b)
            loss_s = loss_score_fn(score_pred, Y_score_b)
            loss_n = loss_next_fn(next_logits_flat, next_labels_flat)

            loss = W_WIN * loss_w + W_SCORE * loss_s + W_NEXT * loss_n

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()

            # Win accuracy
            preds = (torch.sigmoid(win_logit) > 0.5).float()
            correct_win += (preds == Y_win_b).sum().item()
            total_samples += len(Y_win_b)

    avg_loss = total_loss / len(loader)
    win_acc  = correct_win / total_samples
    return avg_loss, win_acc

# ---- Training loop ----
print("\n{'Epoch':<8} {'Train Loss':<14} {'Train WinAcc':<16} {'Val Loss':<12} {'Val WinAcc'}")
print("-" * 65)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)

    scheduler.step(val_loss)

    print(f"{epoch:<8} {train_loss:<14.4f} {train_acc:<16.3%} {val_loss:<12.4f} {val_acc:.3%}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "cricketiq_best.pt")

print(f"\nBest val loss: {best_val_loss:.4f}  |  Model saved to cricketiq_best.pt")

# ---- Final test evaluation ----
model.load_state_dict(torch.load("cricketiq_best.pt"))
test_loss, test_acc = run_epoch(test_loader, training=False)
print(f"\nTest Loss: {test_loss:.4f}  |  Test Win Accuracy: {test_acc:.3%}")

Train: 5403 | Val: 1081 | Test: 721

{'Epoch':<8} {'Train Loss':<14} {'Train WinAcc':<16} {'Val Loss':<12} {'Val WinAcc'}
-----------------------------------------------------------------
1        0.9714         62.706%          0.7962       77.983%
2        0.7815         78.345%          0.7315       79.001%
3        0.7503         79.160%          0.7326       79.093%
4        0.7801         79.622%          0.7747       78.723%
5        0.7015         81.640%          0.6867       81.684%
6        0.6748         81.714%          0.6498       81.776%
7        0.6864         81.862%          0.6815       81.776%
8        0.6768         81.825%          0.7824       77.798%
9        0.6795         81.085%          0.6674       81.314%
10       0.6586         82.399%          0.6452       82.054%
11       0.6532         82.195%          0.6745       81.961%
12       0.6571         82.621%          0.6592       81.314%
13       0.6482         82.288%          0.6378       82.239%
14    

In [5]:
# ============================================================
# CELL 5: Manual Inference — Type your match scenario
# ============================================================

import torch
import numpy as np

# ---- Constants (must match Cell 2) ----
MAX_BALLS   = 120
MAX_SCORE   = 260.0
MAX_WICKETS = 10.0
NEXT_BALL_LABELS = {0: "Dot / Other", 1: "Boundary (4 or 6)", 2: "Wicket"}

# ---- Load best model (in case this cell is run standalone) ----
model.load_state_dict(torch.load("cricketiq_best.pt", map_location=device))
model.eval()

def predict_scenario(
    runs_so_far,
    wickets_so_far,
    balls_bowled,
    target_score=0,
    inning=1
):
    if balls_bowled < 1:
        balls_bowled = 1

    balls_remaining = MAX_BALLS - balls_bowled
    run_rate        = runs_so_far / balls_bowled
    runs_needed     = max(target_score - runs_so_far, 0) if inning == 2 else 0
    req_run_rate    = (runs_needed / balls_remaining) if (inning == 2 and balls_remaining > 0) else 0.0

    features = np.array([
        balls_bowled                       / MAX_BALLS,
        runs_so_far                        / MAX_SCORE,
        wickets_so_far                     / MAX_WICKETS,
        balls_remaining                    / MAX_BALLS,
        min(run_rate, 5)                   / 5.0,
        target_score                       / MAX_SCORE,
        min(runs_needed, MAX_SCORE)        / MAX_SCORE,
        min(req_run_rate, 10)              / 10.0,
        float(inning - 1),
    ], dtype=np.float32)

    seq = np.zeros((MAX_BALLS, len(features)), dtype=np.float32)
    seq[:balls_bowled] = features

    X_input = torch.tensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        win_logit, score_pred, next_logits = model(X_input)

    win_prob = torch.sigmoid(win_logit).item()

    # Hard floor: predicted score can never be less than runs already scored
    predicted_final_score = max(score_pred.item() * MAX_SCORE, float(runs_so_far))

    ball_idx   = min(balls_bowled, MAX_BALLS - 1)
    next_probs = torch.softmax(next_logits[0, ball_idx], dim=0).cpu().numpy()
    next_class = int(np.argmax(next_probs))

    return win_prob, predicted_final_score, next_class, next_probs


# ================================================================
# INTERACTIVE MANUAL TEST — modify the values below and run
# ================================================================

print("=" * 55)
print("CricketIQ — Ball-by-Ball Match Predictor")
print("=" * 55)

# ---- CHANGE THESE VALUES TO YOUR MATCH SCENARIO ----
scenario = {
    "runs_so_far"    : 100,
    "wickets_so_far" : 2,
    "balls_bowled"   : 80,
    "target_score"   : 160,
    "inning"         : 2,
}
# ----------------------------------------------------

win_prob, final_score, next_class, next_probs = predict_scenario(**scenario)

# Hard floor applied again at display level
final_score = max(final_score, float(scenario['runs_so_far']))

print(f"\n📋 Match Scenario:")
print(f"   Innings        : {scenario['inning']}")
print(f"   Balls bowled   : {scenario['balls_bowled']} / 120  ({scenario['balls_bowled']//6} overs {scenario['balls_bowled']%6} balls)")
print(f"   Runs scored    : {scenario['runs_so_far']}")
print(f"   Wickets fallen : {scenario['wickets_so_far']}")
if scenario['inning'] == 2:
    print(f"   Target score   : {scenario['target_score']}")
    print(f"   Runs needed    : {max(scenario['target_score'] - scenario['runs_so_far'], 0)} off {120 - scenario['balls_bowled']} balls")

print(f"\n🔮 Predictions:")
print(f"   Win Probability       : {win_prob:.1%}  (batting team)")
print(f"   Predicted Final Score : {final_score:.0f} runs")
print(f"   Next Ball Outcome     : {NEXT_BALL_LABELS[next_class]}")
print(f"\n   Next Ball Class Probabilities:")
for i, label in NEXT_BALL_LABELS.items():
    bar = "█" * int(next_probs[i] * 30)
    print(f"     {label:<22}: {next_probs[i]:.1%}  {bar}")
print("=" * 55)


# ---- Bonus: step-through mode ----
print("\n\n📊 Over-by-over win probability trace:")
print(f"{'Ball':>6} {'Runs':>6} {'Wkts':>5} {'Win Prob':>10} {'Next Ball':>20}")
print("-" * 55)

for ball in range(6, 121, 6):
    sim_runs = int(ball * 1.25)
    sim_wkts = min(ball // 30, 10)
    wp, fs, nc, _ = predict_scenario(sim_runs, sim_wkts, ball, target_score=160, inning=2)
    print(f"{ball:>6} {sim_runs:>6} {sim_wkts:>5} {wp:>10.1%} {NEXT_BALL_LABELS[nc]:>20}")

CricketIQ — Ball-by-Ball Match Predictor

📋 Match Scenario:
   Innings        : 2
   Balls bowled   : 80 / 120  (13 overs 2 balls)
   Runs scored    : 100
   Wickets fallen : 2
   Target score   : 160
   Runs needed    : 60 off 40 balls

🔮 Predictions:
   Win Probability       : 48.2%  (batting team)
   Predicted Final Score : 100 runs
   Next Ball Outcome     : Dot / Other

   Next Ball Class Probabilities:
     Dot / Other           : 84.2%  █████████████████████████
     Boundary (4 or 6)     : 11.4%  ███
     Wicket                : 4.5%  █


📊 Over-by-over win probability trace:
  Ball   Runs  Wkts   Win Prob            Next Ball
-------------------------------------------------------
     6      7     0      48.2%          Dot / Other
    12     15     0      48.2%          Dot / Other
    18     22     0      48.2%          Dot / Other
    24     30     0      48.2%          Dot / Other
    30     37     1      47.9%          Dot / Other
    36     45     1      47.9%          D

In [6]:
# ============================================================
# CELL 6: Gradio UI for Manual CricketIQ Testing
# ============================================================

%pip install gradio -q

import gradio as gr
import torch
import numpy as np

# ---- Constants (must match Cell 2) ----
MAX_BALLS        = 120
MAX_SCORE        = 260.0
MAX_WICKETS      = 10.0
NEXT_BALL_LABELS = {0: "Dot / Other", 1: "Boundary (4 or 6)", 2: "Wicket"}

# ---- Make sure best model is loaded ----
model.load_state_dict(torch.load("cricketiq_best.pt", map_location=device))
model.eval()

# ---- Core prediction function ----
def predict_scenario(
    runs_so_far,
    wickets_so_far,
    balls_bowled,
    target_score,
    inning,
    toss_won_batting
):
    balls_bowled    = max(1, int(balls_bowled))
    balls_remaining = MAX_BALLS - balls_bowled
    run_rate        = runs_so_far / balls_bowled
    runs_needed     = max(target_score - runs_so_far, 0) if inning == 2 else 0
    req_run_rate    = (runs_needed / balls_remaining) if (inning == 2 and balls_remaining > 0) else 0.0

    features = np.array([
        balls_bowled                       / MAX_BALLS,
        runs_so_far                        / MAX_SCORE,
        wickets_so_far                     / MAX_WICKETS,
        balls_remaining                    / MAX_BALLS,
        min(run_rate, 5)                   / 5.0,
        target_score                       / MAX_SCORE,
        min(runs_needed, MAX_SCORE)        / MAX_SCORE,
        min(req_run_rate, 10)              / 10.0,
        float(inning - 1),
    ], dtype=np.float32)

    seq = np.zeros((MAX_BALLS, len(features)), dtype=np.float32)
    seq[:balls_bowled] = features

    X_input = torch.tensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        win_logit, score_pred, next_logits = model(X_input)

    win_prob_raw = torch.sigmoid(win_logit).item()
    toss_bias    = 0.03 if toss_won_batting == 1 else -0.03
    win_prob     = float(np.clip(win_prob_raw + toss_bias, 0.01, 0.99))

    # Hard floor: predicted score can never be less than runs already scored
    predicted_final_score = max(score_pred.item() * MAX_SCORE, float(runs_so_far))

    ball_idx   = min(balls_bowled, MAX_BALLS - 1)
    next_probs = torch.softmax(next_logits[0, ball_idx], dim=0).cpu().numpy()
    next_class = int(np.argmax(next_probs))

    return win_prob, predicted_final_score, next_class, next_probs


# ---- Gradio handler ----
def gradio_predict(runs_so_far, wickets_so_far, balls_bowled,
                   target_score, inning_str, toss_str):

    inning           = 1 if inning_str == "1st Innings" else 2
    toss_won_batting = 1 if toss_str == "Yes" else 0

    if balls_bowled <= 0:
        return "❌ Balls bowled must be at least 1.", "", "", "", ""
    if wickets_so_far < 0 or wickets_so_far > 10:
        return "❌ Wickets must be between 0 and 10.", "", "", "", ""
    if inning == 1:
        target_score = 0

    win_prob, final_score, next_class, next_probs = predict_scenario(
        runs_so_far, wickets_so_far, balls_bowled,
        target_score, inning, toss_won_batting
    )

    # Hard floor applied again at display level
    final_score = max(final_score, float(runs_so_far))

    overs_done = balls_bowled // 6
    balls_over = balls_bowled % 6
    balls_left = MAX_BALLS - balls_bowled

    # 1. Match snapshot
    snapshot_lines = [
        f"Innings       : {inning_str}",
        f"Progress      : {overs_done}.{balls_over} overs  ({balls_bowled}/120 balls)",
        f"Score         : {int(runs_so_far)}/{int(wickets_so_far)}",
        f"Balls left    : {balls_left}",
    ]
    if inning == 2:
        runs_needed = max(int(target_score) - int(runs_so_far), 0)
        rr_needed   = runs_needed / balls_left if balls_left > 0 else 999
        snapshot_lines += [
            f"Target        : {int(target_score)}",
            f"Runs needed   : {runs_needed} off {balls_left} balls",
            f"Required RR   : {rr_needed:.2f} per ball  ({rr_needed*6:.2f} per over)",
        ]
    snapshot_lines.append(f"Toss won batting : {'Yes' if toss_won_batting else 'No'}")
    snapshot = "\n".join(snapshot_lines)

    # 2. Win probability
    lose_prob = 1.0 - win_prob
    win_bar   = "🟩" * int(win_prob  * 20)
    lose_bar  = "🟥" * int(lose_prob * 20)
    win_out   = (
        f"Batting Team Wins : {win_prob:.1%}  {win_bar}\n"
        f"Bowling Team Wins : {lose_prob:.1%}  {lose_bar}"
    )

    # 3. Predicted final score — floor guaranteed
    score_out = f"Predicted Final Score : {final_score:.0f} runs"
    if inning == 2:
        result_guess = "✅ Chase likely successful" if win_prob > 0.5 else "❌ Chase likely falls short"
        score_out += f"\n{result_guess}"

    # 4. Next ball
    dot_p, bdry_p, wkt_p = next_probs[0], next_probs[1], next_probs[2]
    next_out = (
        f"Most Likely  : {NEXT_BALL_LABELS[next_class]}\n\n"
        f"Dot / Other  : {dot_p:.1%}  {'█' * int(dot_p  * 30)}\n"
        f"Boundary     : {bdry_p:.1%}  {'█' * int(bdry_p * 30)}\n"
        f"Wicket       : {wkt_p:.1%}  {'█' * int(wkt_p  * 30)}"
    )

    # 5. Commentary
    if win_prob > 0.75:
        commentary = "🔥 Batting team is in firm control."
    elif win_prob > 0.55:
        commentary = "📈 Batting team has the upper hand."
    elif win_prob > 0.45:
        commentary = "⚖️  It's a very close contest!"
    elif win_prob > 0.25:
        commentary = "📉 Bowling team is on top."
    else:
        commentary = "🛑 Batting team in serious trouble."

    return snapshot, win_out, score_out, next_out, commentary


# ---- Gradio UI ----
with gr.Blocks(title="CricketIQ", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏏 CricketIQ — Ball-by-Ball Match Outcome Predictor
    Fill in the current match situation and click **Predict** to get live predictions.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Match Inputs")

            runs_input    = gr.Number(label="Runs Scored So Far",  value=100, precision=0)
            wickets_input = gr.Number(label="Wickets Fallen",       value=2,   precision=0)
            balls_input   = gr.Number(label="Balls Bowled (0–120)", value=80,  precision=0)
            target_input  = gr.Number(
                label="Target Score (2nd innings only; set 0 for 1st)",
                value=160, precision=0
            )
            inning_input  = gr.Radio(
                choices=["1st Innings", "2nd Innings"],
                value="2nd Innings",
                label="Innings"
            )
            toss_input    = gr.Radio(
                choices=["Yes", "No"],
                value="Yes",
                label="Did Batting Team Win the Toss?"
            )
            predict_btn = gr.Button("🔮 Predict", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 📊 Predictions")

            commentary_out = gr.Textbox(label="📣 Commentary",          lines=1, interactive=False)
            snapshot_out   = gr.Textbox(label="📋 Match Snapshot",       lines=9, interactive=False)
            win_out        = gr.Textbox(label="🏆 Win Probability",      lines=3, interactive=False)
            score_out      = gr.Textbox(label="🎯 Predicted Final Score", lines=3, interactive=False)
            next_out       = gr.Textbox(label="⚡ Next Ball Prediction",  lines=5, interactive=False)

    predict_btn.click(
        fn=gradio_predict,
        inputs=[runs_input, wickets_input, balls_input,
                target_input, inning_input, toss_input],
        outputs=[snapshot_out, win_out, score_out, next_out, commentary_out]
    )

    gr.Markdown("""
    ---
    **How to use:**
    - **1st innings**: Innings = 1st, Target = 0. Predicts final score & next ball.
    - **2nd innings**: Innings = 2nd, enter the target. Win probability reflects the chase.
    - Toss applies a small ±3% soft adjustment to win probability.
    """)

demo.launch(share=True)

Note: you may need to restart the kernel to use updated packages.


d:\PY Projects\CricIQ\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\muham\AppData\Local\Temp\ipykernel_31184\2362300990.py:154: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="CricketIQ", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
